#  ConnectaTel — Análisis de Clientes 2024
## ¿Qué segmentos de clientes son más valiosos y qué patrones de uso los caracterizan?

**Proyecto Integrador — Storytelling con Datos**  
Este informe integra limpieza, transformación, análisis y visualización de datos en una narrativa cohesiva dirigida al equipo directivo de ConnectaTel.

### Contexto
ConnectaTel es una empresa de telecomunicaciones que opera en seis ciudades de Latinoamérica (Bogotá, Medellín, Cali, CDMX, GDL, MTY). Cuenta con 4,000 clientes registrados, dos planes de servicio (Básico y Premium), y ha generado 40,000 eventos de uso entre llamadas y mensajes de texto.

El análisis busca responder con datos concretos: **¿quiénes son nuestros mejores clientes y qué los distingue?**

In [ ]:
# Instalaciones y librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Estilo de gráficas
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.titleweight'] = 'bold'

print('Librerías correctamente instaladas')


In [ ]:
# Carga de datos
from pathlib import Path

base_dir = Path('C:/Users/silvi/Desktop/Proyectos/analisis_datos').resolve()
usage_file = base_dir / 'usage_connectatel.csv'
users_file = base_dir / 'users_connectatel.csv'

print('base_dir =', base_dir)
print('usage_file exists =', usage_file.exists())
print('users_file exists =', users_file.exists())

usage = pd.read_csv(usage_file)
users = pd.read_csv(users_file)

print('\nusage shape:', usage.shape)
print('users shape:', users.shape)
print('\n--- usage (primeras 3 filas) ---')
print(usage.head(3).to_string(index=False))
print('\n--- users (primeras 3 filas) ---')
print(users.head(3).to_string(index=False))

---
##  Limpieza del dataset

Antes de analizar, limpiamos los datos:
- La columna `city` contiene el valor `'?'` para ciudades desconocidas — lo reemplazamos por `NaN`.
- La columna `date` en usage se convierte a datetime.
- Valores negativos en `age` se excluyen del análisis demográfico (son errores de captura).

La razón de estos pasos es garantizar que los promedios y conteos no estén contaminados por datos incorrectos.

In [ ]:
# Limpieza
users['city'] = users['city'].replace('?', np.nan)
usage['date'] = pd.to_datetime(usage['date'], errors='coerce')

print('Nulos en users:')
print(users.isnull().sum())
print()
print('Nulos en usage:')
print(usage.isnull().sum())

---
##  Bloque 0 — Agrupación y análisis

### Paso 1: Construcción del dataset combinado (df_connectatel)

###Este punto ya lo tenia en la actividad anterior pero decidí mejorarlo

In [30]:
# Agregación por usuario
calls_agg = usage[usage['type']=='call'].groupby('user_id')['duration'].sum().rename('total_minutes')
texts_agg = usage[usage['type']=='text'].groupby('user_id')['length'].count().rename('total_texts')

df = users.merge(calls_agg, on='user_id', how='left').merge(texts_agg, on='user_id', how='left')
df['total_minutes'] = df['total_minutes'].fillna(0)
df['total_texts'] = df['total_texts'].fillna(0)

print(f'df_connectatel shape: {df.shape}')
display(df.head(3))

df_connectatel shape: (4000, 17)


,user_id,first_name,last_name,age,city,reg_date,plan,churn_date,messages_included,gb_per_month,minutes_included,usd_monthly_pay,usd_per_gb,usd_per_message,usd_per_minute,total_minutes,total_texts
0,10000,Carlos,Garcia,38,Medellín,2022-01-01 00:00:00.000000000,Basico,NaN,100,5,100,12,1.2,0.08,0.1,23.70,7.0
1,10001,Mateo,Torres,53,?,2022-01-01 06:34:17.914478619,Basico,NaN,100,5,100,12,1.2,0.08,0.1,33.18,5.0
2,10002,Sofia,Ramirez,57,CDMX,2022-01-01 13:08:35.828957239,Basico,NaN,100,5,100,12,1.2,0.08,0.1,10.74,5.0


### 1 — Nuevas columnas calculadas con `assign()`

Creo tres columnas nuevas que tienen sentido de negocio real:

- **`rev_calls`**: ingreso estimado generado por llamadas (`total_minutes × usd_per_minute`)
- **`rev_texts`**: ingreso estimado por mensajes (`total_texts × usd_per_message`)
- **`total_revenue`**: ingreso total = tarifa mensual + excedentes de llamadas + excedentes de mensajes
- **`is_active`**: booleano que indica si el cliente NO tiene fecha de churn (sigue activo)
- **`overage_minutes`**: minutos que exceden el plan incluido — indica clientes que consumen más de lo contratado

Estas columnas permiten calcular el valor real de cada cliente, y asi no solo tener un tarifario general

In [31]:
df = df.assign(
    rev_calls    = lambda x: x['total_minutes'] * x['usd_per_minute'],
    rev_texts    = lambda x: x['total_texts']   * x['usd_per_message'],
    total_revenue= lambda x: x['usd_monthly_pay'] + (x['total_minutes'] * x['usd_per_minute']) + (x['total_texts'] * x['usd_per_message']),
    is_active    = lambda x: x['churn_date'].isna(),
    overage_minutes = lambda x: (x['total_minutes'] - x['minutes_included']).clip(lower=0)
)

print('Nuevas columnas creadas:')
print(df[['user_id','plan','total_minutes','rev_calls','rev_texts','total_revenue','is_active','overage_minutes']].head(5).to_string())

Nuevas columnas creadas:
   user_id     plan  total_minutes  rev_calls  rev_texts  total_revenue  is_active  overage_minutes
0    10000   Basico          23.70     2.3700       0.56        14.9300       True              0.0
1    10001   Basico          33.18     3.3180       0.40        15.7180       True              0.0
2    10002   Basico          10.74     1.0740       0.40        13.4740       True              0.0
3    10003  Premium           8.99     0.6293       0.55        26.1793       True              0.0
4    10004   Basico           8.01     0.8010       0.32        13.1210       True              0.0


### 2 — Filtrado con condiciones combinadas

Filtro el subconjunto de clientes **Premium activos con más de 30 minutos de uso** representan los clientes de mayor engagement en el segmento más valioso.

In [32]:
premium_activos_alto_uso = df[
    (df['plan'] == 'Premium') &
    (df['is_active'] == True) &
    (df['total_minutes'] > 30)
]

print(f'Filas en el subconjunto: {len(premium_activos_alto_uso)}')
print(f'Ingreso promedio de este grupo: ${premium_activos_alto_uso["total_revenue"].mean():.2f} USD')
print(f'Ingreso promedio general: ${df["total_revenue"].mean():.2f} USD')
display(premium_activos_alto_uso[['user_id','city','plan','total_minutes','total_revenue']].head())

Filas en el subconjunto: 353
Ingreso promedio de este grupo: $28.31 USD
Ingreso promedio general: $18.99 USD


,user_id,city,plan,total_minutes,total_revenue
7,10007,Medellín,Premium,30.23,27.2661
27,10027,MTY,Premium,84.77,31.3339
33,10033,GDL,Premium,42.50,28.0750
46,10046,CDMX,Premium,32.39,27.5673
58,10058,MTY,Premium,37.73,27.7411


**Markdown:** Hay **clientes de alto valor operativo** Premium activos con más de 30 minutos de uso mensual. Su ingreso promedio es notablemente mayor al general, confirmando que el volumen de uso en llamadas es un predictor relevante del valor del cliente.

### 3  Ordenamiento por múltiples columnas

In [33]:
df_sorted = df.sort_values(['total_revenue', 'total_minutes'], ascending=[False, False])

cols = ['user_id','plan','city','total_minutes','total_texts','total_revenue']
print('=== TOP 5 (mayor ingreso) ===')
display(df_sorted[cols].head(5))
print('=== BOTTOM 5 (menor ingreso) ===')
display(df_sorted[cols].tail(5))

=== TOP 5 (mayor ingreso) ===


,user_id,plan,city,total_minutes,total_texts,total_revenue
585,10585,Premium,GDL,153.02,5.0,35.9614
534,10534,Premium,CDMX,153.06,3.0,35.8642
216,10216,Premium,GDL,148.76,4.0,35.6132
1589,11589,Premium,Bogotá,136.55,3.0,34.7085
3493,13493,Premium,Bogotá,131.88,6.0,34.5316


=== BOTTOM 5 (menor ingreso) ===


,user_id,plan,city,total_minutes,total_texts,total_revenue
2058,12058,Basico,NaN,1.85,0.0,12.185
1956,11956,Basico,CDMX,0.11,2.0,12.171
1084,11084,Basico,GDL,0.00,2.0,12.160
2807,12807,Basico,CDMX,0.25,1.0,12.105
1082,11082,Basico,CDMX,0.00,0.0,12.000


**Interpretación:** Los 5 clientes más rentables son todos del plan **Premium** con uso intensivo de llamadas (>130 min). Los 5 menos rentables tienen ingreso casi únicamente por tarifa base (pocos o ningún minuto de llamada). El patrón es claro: el uso activo del servicio de voz correlaciona con mayor ingreso.

### 4  `idxmax()` e `idxmin()` sobre columna numérica

In [34]:
idx_max = df['total_revenue'].idxmax()
idx_min = df['total_revenue'].idxmin()

print('=== Cliente con MAYOR ingreso ===')
display(df.loc[idx_max, ['user_id','plan','city','total_minutes','total_texts','total_revenue']].to_frame().T)

print('=== Cliente con MENOR ingreso ===')
display(df.loc[idx_min, ['user_id','plan','city','total_minutes','total_texts','total_revenue']].to_frame().T)

=== Cliente con MAYOR ingreso ===


,user_id,plan,city,total_minutes,total_texts,total_revenue
585,10585,Premium,GDL,153.02,5.0,35.9614


=== Cliente con MENOR ingreso ===


,user_id,plan,city,total_minutes,total_texts,total_revenue
1082,11082,Basico,CDMX,0.0,0.0,12.0


** Interpretación:** El cliente de mayor ingreso es Premium, de GDL, con >153 minutos de llamadas. El de menor ingreso tiene 0 minutos y 0 mensajes — es un cliente inactivo en uso pero que sigue pagando la tarifa base. Esto sugiere una oportunidad para activar clientes dormidos.

### 5 Agrupación por una columna categórica

In [35]:
por_plan = df.groupby('plan').agg(
    ingreso_promedio=('total_revenue', 'mean'),
    ingreso_maximo  =('total_revenue', 'max'),
    total_clientes  =('user_id', 'count'),
    minutos_promedio=('total_minutes', 'mean'),
    mensajes_promedio=('total_texts', 'mean')
).round(2)

display(por_plan)

,ingreso_promedio,ingreso_maximo,total_clientes,minutos_promedio,mensajes_promedio
plan,,,,,
Basico,14.70,27.57,2595,22.57,5.53
Premium,26.91,35.96,1405,23.31,5.52


** Markdown:** El plan **Básico** tiene el mayor conteo (2,595 clientes) pero el **Premium** tiene el mayor promedio de ingreso ($26.91 vs $14.70). Son grupos diferentes en tamaño y en valor unitario. Esto importa: tener más clientes Básico no compensa la diferencia de valor por cabeza, pero en volumen total ambos planes contribuyen cantidades similares al ingreso total de la empresa.

### 6 Agrupación por dos columnas categóricas

In [ ]:
# Reemplazar los valores '?' en city por 'Desconocida'
df['city'] = df['city'].replace('?', 'Desconocida')

# Volvemos a agrupar
por_plan_ciudad = df.groupby(['plan','city']).agg(
    ingreso_promedio=('total_revenue', 'mean'),
    total_clientes=('user_id', 'count')
).round(2).sort_values('ingreso_promedio', ascending=False)

display(por_plan_ciudad.head(12))


**Markdown:** La combinación **Premium + Cali** lidera en ingreso promedio. Sin embargo, hay combinaciones con pocos registros (algunas ciudades pequeñas en plan Premium) — hay que tener precaución al extrapolar conclusiones de grupos con menos de 50 clientes, ya que los promedios son más sensibles a valores atípicos.

### 7 Comparación de dos grupos: Premium vs Básico

In [39]:
premium_df = df[df['plan'] == 'Premium']
basico_df  = df[df['plan'] == 'Basico']

print('=== Premium ===')
display(premium_df[['total_revenue','total_minutes','total_texts']].describe().round(2))
print('=== Básico ===')
display(basico_df[['total_revenue','total_minutes','total_texts']].describe().round(2))

=== Premium ===


,total_revenue,total_minutes,total_texts
count,1405.00,1405.00,1405.00
mean,26.91,23.31,5.52
std,1.17,16.70,2.35
min,25.10,0.00,0.00
25%,26.09,11.58,4.00
50%,26.68,20.30,5.00
75%,27.53,32.19,7.00
max,35.96,153.06,16.00


=== Básico ===


,total_revenue,total_minutes,total_texts
count,2595.00,2595.00,2595.00
mean,14.70,22.57,5.53
std,1.66,16.53,2.36
min,12.00,0.00,0.00
25%,13.52,10.89,4.00
50%,14.40,19.51,5.00
75%,15.51,30.58,7.00
max,27.57,155.69,17.00


**Markdown :** La diferencia más significativa entre grupos está en `total_revenue` (media: $26.91 vs $14.70 — 83% más alto en Premium). En contraste, `total_minutes` y `total_texts` son prácticamente idénticos entre grupos. Esta es la paradoja central del dataset: los clientes Premium pagan más pero **no usan más** el servicio.

### 8  Ranking de ciudades por ingreso promedio (Top 5 y Bottom 5)

In [40]:
ranking_ciudades = df.dropna(subset=['city']).groupby('city')['total_revenue'].mean().sort_values(ascending=False).round(2)

print('=== TOP 5 ciudades por ingreso promedio ===')
print(ranking_ciudades.head(5).to_string())
print()
print('=== BOTTOM 5 ciudades por ingreso promedio ===')
print(ranking_ciudades.tail(5).to_string())

=== TOP 5 ciudades por ingreso promedio ===
city
Cali        19.33
Medellín    19.08
Bogotá      19.02
GDL         18.93
CDMX        18.93

=== BOTTOM 5 ciudades por ingreso promedio ===
city
Bogotá         19.02
GDL            18.93
CDMX           18.93
Desconocida    18.75
MTY            18.64


** Markdown :** La diferencia entre la ciudad con mayor ingreso promedio (Cali, $19.33) y la de menor (MTY, $18.64) es de apenas $0.69 USD. Las 6 ciudades forman un grupo muy homogéneo — esto sugiere que la estrategia comercial puede ser uniforme a nivel geográfico, sin necesidad de diferenciación regional significativa.

---
## Paso 1 — Visualizaciones por segmentos

Generamos al menos 4 gráficas que responden visualmente la pregunta central del análisis.

In [ ]:
# GRÁFICA 1: Distribución de ingresos por plan
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Boxplot
sns.boxplot(data=df, x='plan', y='total_revenue', palette={'Basico':'#34d399','Premium':'#a78bfa'}, ax=axes[0])
axes[0].set_title('Distribución de ingresos por plan')
axes[0].set_xlabel('Plan')
axes[0].set_ylabel('Ingreso total estimado (USD)')

# Barplot promedio
promedios = df.groupby('plan')['total_revenue'].mean()
colors = ['#34d399' if p == 'Basico' else '#a78bfa' for p in promedios.index]
axes[1].bar(promedios.index, promedios.values, color=colors, width=0.5, edgecolor='none')
for i, (plan, val) in enumerate(promedios.items()):
    axes[1].text(i, val + 0.3, f'${val:.2f}', ha='center', fontweight='bold', fontsize=12)
axes[1].set_title('Ingreso promedio por plan')
axes[1].set_xlabel('Plan')
axes[1].set_ylabel('USD promedio')
axes[1].set_ylim(0, 32)

plt.suptitle('Gráfica 1 — Valor económico por segmento de plan', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

** Markdown :** El plan Premium genera un ingreso promedio de **$26.91 USD**, 83% más que el plan Básico ($14.70 USD). La distribución es más amplia en Premium, lo que indica mayor variabilidad — hay clientes Premium que generan hasta $35+ USD mensuales.

In [ ]:
# GRÁFICA 2: Uso de minutos y mensajes — comparativa de planes
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.boxplot(data=df, x='plan', y='total_minutes', palette={'Basico':'#34d399','Premium':'#a78bfa'}, ax=axes[0])
axes[0].set_title('Minutos de llamada por plan')
axes[0].set_xlabel('Plan'); axes[0].set_ylabel('Minutos totales')

sns.boxplot(data=df, x='plan', y='total_texts', palette={'Basico':'#34d399','Premium':'#a78bfa'}, ax=axes[1])
axes[1].set_title('Mensajes de texto por plan')
axes[1].set_xlabel('Plan'); axes[1].set_ylabel('Número de mensajes')

plt.suptitle('Gráfica 2 — La paradoja del uso: mismos hábitos, diferente precio', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

** Markdown  (la paradoja):** Los minutos y mensajes son **prácticamente idénticos** entre planes (Básico: 22.6 min, 5.53 msgs; Premium: 23.3 min, 5.52 msgs). La diferencia de ingresos del 83% no se explica por mayor consumo — viene de la **tarifa base mensual** del plan Premium.

In [ ]:
# GRÁFICA 3: Ingreso promedio y número de clientes por ciudad
ciudad_stats = df.dropna(subset=['city']).groupby('city').agg(
    ingreso_prom=('total_revenue','mean'),
    n_clientes=('user_id','count')
).sort_values('ingreso_prom', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors_city = sns.color_palette('muted', len(ciudad_stats))
axes[0].barh(ciudad_stats.index, ciudad_stats['ingreso_prom'], color=colors_city, edgecolor='none')
axes[0].set_xlabel('Ingreso promedio (USD)')
axes[0].set_title('Ingreso promedio por ciudad')
axes[0].set_xlim(18, 20)

ciudad_count = df.dropna(subset=['city']).groupby('city')['user_id'].count().sort_values(ascending=True)
axes[1].barh(ciudad_count.index, ciudad_count.values, color=colors_city[::-1], edgecolor='none')
axes[1].set_xlabel('Número de clientes')
axes[1].set_title('Número de clientes por ciudad')

plt.suptitle('Gráfica 3 — Distribución geográfica: ingresos y volumen', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

** Markdown :** Bogotá es la ciudad con más clientes (808), pero Cali lidera en ingreso promedio ($19.33 USD). La diferencia entre todas las ciudades es menor a $0.70 USD — el comportamiento de los clientes es notablemente homogéneo en toda la red.

In [ ]:
# GRÁFICA 4: Churn por plan y composición de ingresos
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Churn por plan
churn_plan = df.groupby('plan')['is_active'].apply(lambda x: (~x).mean() * 100)
colors_churn = ['#34d399','#a78bfa']
axes[0].bar(churn_plan.index, churn_plan.values, color=colors_churn, width=0.5, edgecolor='none')
for i, (plan, val) in enumerate(churn_plan.items()):
    axes[0].text(i, val + 0.1, f'{val:.1f}%', ha='center', fontweight='bold', fontsize=12)
axes[0].set_title('Tasa de churn por plan')
axes[0].set_xlabel('Plan'); axes[0].set_ylabel('% de clientes con churn')
axes[0].set_ylim(0, 18)

# Composición de ingresos totales
ingresos_totales = df.groupby('plan')['total_revenue'].sum()
axes[1].pie(ingresos_totales.values, labels=ingresos_totales.index,
            autopct='%1.1f%%', colors=['#34d399','#a78bfa'],
            startangle=90, wedgeprops=dict(edgecolor='none'))
axes[1].set_title('Composición del ingreso total por plan')

plt.suptitle('Gráfica 4 — Churn y distribución de ingresos', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

** Markdown:** La tasa de churn es similar en ambos planes (Básico: 11.5%, Premium: 12.0%), lo que indica que el precio no es el factor principal de abandono. En cuanto a ingresos, los planes están casi equilibrados: Premium aporta el 49.8% del total pese a tener solo el 35.1% de los clientes.

In [ ]:
# GRÁFICA 5 (extra): Scatter de minutos vs ingreso, coloreado por plan
fig, ax = plt.subplots(figsize=(10, 6))

for plan, color in [('Basico', '#34d399'), ('Premium', '#a78bfa')]:
    subset = df[df['plan'] == plan]
    ax.scatter(subset['total_minutes'], subset['total_revenue'],
               alpha=0.3, color=color, label=plan, s=20)

ax.set_xlabel('Minutos totales de llamada')
ax.set_ylabel('Ingreso total estimado (USD)')
ax.set_title('Gráfica 5 — Relación entre uso de voz e ingreso por plan')
ax.legend(title='Plan')
plt.tight_layout()
plt.show()

** Markdown:** El scatter confirma la separación entre planes: los puntos Premium (morado) se ubican sistemáticamente más arriba que los Básico (verde) para el mismo número de minutos. La pendiente positiva indica que más minutos sí genera más ingreso, pero el nivel base es determinado principalmente por el plan elegido.

---
## Paso 2 — Storytelling: ¿Qué segmentos son más valiosos?

### La historia de los clientes de ConnectaTel

----
**Capítulo 1: ConnectaTel, cuatro mil historias de conexión**
ConnectaTel tiene unos 4,000 clientes. La mayoría sigue con nosotros: casi 9 de cada 10 personas que se unieron todavía usan el servicio. Entre todos han hecho más de 40,000 interacciones, y lo curioso es que prefieren mandar mensajes antes que hablar por teléfono. Más de la mitad de las veces, nuestros clientes escriben en lugar de llamar. Bogotá, Ciudad de México y Medellín son las ciudades con más usuarios, y juntos generan ingresos que superan los 75 mil dólares.

Entre nuestros clientes hay un grupo que llamamos los clientes mágicos. No sabemos de qué ciudad son, porque sus datos quedaron incompletos. En un mundo lleno de información, ellos eligieron permanecer invisibles. Sin embargo, siguen enviando mensajes y haciendo llamadas. Son un recordatorio de que la conexión real no siempre depende de los datos que tenemos, sino de las acciones que las personas deciden hacer.---


**Capítulo 2: Los clientes de alto valor operativo — menos, pero más rentables**

Aunque la mayoría de la gente está en el plan Básico, los que tienen el plan Premium son los que más dinero dejan. Cada cliente Premium paga casi el doble que uno Básico, aunque son menos. Por eso decimos que son los clientes de alto valor operativo: no son tantos, pero sostienen gran parte de los ingresos de la empresa. En ciudades como Guadalajara y Bogotá hay varios de los clientes más rentables, que pagan más de 34 dólares al mes.


**Capítulo 3: La gran paradoja — mismo uso, diferente precio**

Lo curioso es que los clientes Premium y los Básico usan el servicio casi igual. Hablan y mandan mensajes en cantidades muy parecidas. La diferencia está en lo que pagan: los Premium tienen una tarifa más alta, no porque usen más, sino porque eligieron ese plan. Eso significa que muchos clientes Básico podrían pasarse a Premium sin cambiar sus hábitos, y la empresa ganaría más dinero.

**Capítulo 4: Lo que los datos nos piden hacer**

Los datos nos dicen que ConnectaTel tiene una base de clientes estable, pero con mucho potencial. Si logramos que parte de los clientes Básico se pasen a Premium, los ingresos podrían crecer muchísimo. Incluso si solo el 20% hiciera el cambio, serían más de 6 mil dólares adicionales al mes. También es importante cuidar a los que ya tenemos: algunos se han ido, y no es por el precio, sino por la experiencia. Si entendemos mejor por qué se van, podremos retenerlos y seguir creciendo.
